In [ ]:
import os

os.environ["PYKEOPS_VERBOSE"] = "0"
os.environ["KEOPS_VERBOSE"] = "0"

In [ ]:
import sys, os
sys.path.append(os.path.join(os.getcwd(), '../../')) # Add root of repo to import MBM

import warnings
import massbalancemachine as mbm
import logging
from datetime import datetime
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import torch 
import json 

from regions.TF_Europe.scripts.config_TF_Europe import *
from regions.TF_Europe.scripts.dataset import *
from regions.TF_Europe.scripts.plotting import *
from regions.TF_Europe.scripts.models import *
from regions.TF_Europe.scripts.utils import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

print(torch.cuda.is_available())   # True
print(torch.cuda.device_count())   # 0  ← wrong$

if torch.cuda.device_count()==0:
    print("No GPU detected. Using CPU.")
    device = torch.device("cpu")


In [ ]:
GLOB_DIR = Path(cfg.dataPath) / path_cache
BASE_DIR = Path(cfg.dataPath) / path_cache / f"TF_REGION"
print(f"Base directory for data: {BASE_DIR}")

# make sure the base directory exists
os.makedirs(BASE_DIR, exist_ok=True)

In [ ]:
MONTHLY_COLS = [
    't2m',
    'tp',
    'slhf',
    'sshf',
    'ssrd',
    'fal',
    'str',
    'ELEVATION_DIFFERENCE',
]
STATIC_COLS = ['aspect', 'slope', 'svf']

feature_columns = MONTHLY_COLS + STATIC_COLS

# Transform data to monthly format (run or load data):
paths = {
    'era5_climate_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_monthly_averaged_data_EU_US_CANADA.nc"),
    'geopotential_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_geopotential_pressure_EU_US_CANADA.nc")
}

# Check that all these files exists
for key, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required file for {key} not found at {path}")

vois_climate = [
    "t2m",
    "tp",
    "slhf",
    "sshf",
    "ssrd",
    "fal",
    "str",
]

vois_topographical = ["aspect", "slope", "svf"]

# Cross-regional modelling:

### Read stakes datasets:

In [ ]:
"""
Examples of loading data:
# Load Switzerland only
df = load_stakes(cfg, "CH")

# Load all Central Europe (FR+CH+IT+AT when you add them)
df_ceu = load_stakes_for_rgi_region(cfg, "11")

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}
"""

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}
dfs["11"].head()

In [ ]:
rgi_regions = dfs.keys()
num_measurements = {}
for rgi_region in rgi_regions:
    src_code_regions = dfs[rgi_region].SOURCE_CODE.unique()
    for src_code in src_code_regions:
        num_measurements[src_code] = len(
            dfs[rgi_region][dfs[rgi_region].SOURCE_CODE == src_code])
num_measurements

### Monthly datasets:
Build monthly datasets for LSTM. 

In [ ]:
SOURCE_REGIONS = ['CH']
experiment_region_groups = {
    # "CEU": ["FR", "IT_AT", "CH"],
    "HMA": ["CENTRALASIA", "SOUTHASIAWEST", "SOUTHASIAEAST"]
}
paths_multi = {
    "EU_US_CANADA": {
        "era5_climate_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_monthly_averaged_data_EU_US_CANADA.nc"),
        "geopotential_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_geopotential_pressure_EU_US_CANADA.nc"),
    },
    "HMA": {
        "era5_climate_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_monthly_averaged_data_HIGH_MOUNTAIN_ASIA.nc"),
        "geopotential_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_geopotential_pressure_HIGH_MOUNTAIN_ASIA.nc"),
    },
}

XREG_PAIRS = [
    # (source, target)
    ("CH", ["ISL"])
]

In [ ]:
# Step 1: collect all unique codes across all pairs
all_codes = sorted(
    {code.upper()
     for src, tgts in XREG_PAIRS
     for code in [src] + tgts})

run_flag_by_code = {
    'CH': False,
}
# Step 2: compute monthly data once per code
monthly_cache = build_monthly_cache(
    cfg=cfg,
    dfs=dfs,
    paths_multi=paths_multi,
    vois_climate=vois_climate,
    vois_topographical=vois_topographical,
    region_codes=all_codes,
    run_flag_by_code=run_flag_by_code,
)

# Step 3: assemble train/test pairs from cache — no ERA5 reprocessing
res_xreg_by_source, split_info_by_source = prepare_xreg_pairs_from_cache(
    cfg=cfg,
    monthly_cache=monthly_cache,
    xreg_pairs=XREG_PAIRS,
)

In [ ]:
# EXCLUDE_TARGETS = {"CAW", "ACA", "GRL"}  # set to empty set to include all
EXCLUDE_TARGETS = {}
res_all_xreg_by_source = {}

SOURCE_MEMBERS = {
    "CH": ["CH"],
    "NOR": ["NOR"],
    "ISL": ["ISL"],
    "ALA": ["ALA"],
    "CENTRALASIA": ["CENTRALASIA"],
    "SOUTHASIAWEST": ["SOUTHASIAWEST"],
    "SOUTHASIAEAST": ["SOUTHASIAEAST"],
}

# The 4 individual targets we always want (each source excludes itself)
ALL_TARGETS = [
    "CH", "NOR", "ISL", "ALA", "SJM", "CENTRALASIA", "SOUTHASIAWEST",
    "SOUTHASIAEAST"
]

for src_region in SOURCE_REGIONS:
    src_members = SOURCE_MEMBERS[src_region]

    # Exclude the source region itself from targets
    target_codes = [t for t in ALL_TARGETS if t not in src_members]

    res_all_xreg = build_xreg_res_all(
        res_xreg=res_xreg_by_source[src_region],
        target_source_codes=target_codes,
        source_col="SOURCE_CODE",
        key_prefix=f"XREG_{src_region}_TO",
        ch_code=src_region,
        region_groups={},  # no group regions anymore
    )

    # Filter out excluded targets
    if EXCLUDE_TARGETS:
        res_all_xreg = {
            k: v
            for k, v in res_all_xreg.items()
            if not any(tgt in k for tgt in EXCLUDE_TARGETS)
        }

    res_all_xreg_by_source[src_region] = res_all_xreg
    print(f"{src_region} -> keys:", list(res_all_xreg.keys()))

### Finetuning glaciers:

#### Automatic picking:

In [ ]:
CLIMATE_COLS = [
    't2m', 'tp', 'slhf', 'sshf', 'ssrd', 'fal', 'str', 'ELEVATION_DIFFERENCE'
]
TOPO_COLS = ['aspect', 'slope', 'svf']
# define scalars:
scaler_m, scaler_s, scaler_all = build_global_scalers_multi_source_simple(
    res_xreg_by_source,
    monthly_cols=CLIMATE_COLS,
    static_cols=TOPO_COLS,
)

blur_m, blur_s, blur_joint = estimate_global_bandwidths_simple(
    res_xreg_by_source,
    monthly_cols=CLIMATE_COLS,
    static_cols=TOPO_COLS,
    scaler_m=scaler_m,
    scaler_s=scaler_s,
    seed=cfg.seed,
)
bandwidths_m = [blur_m * 0.5, blur_m, blur_m * 2.0]
bandwidths_s = [blur_s * 0.5, blur_s, blur_s * 2.0]
print(
    f"Estimated blurs: blur_m={blur_m:.4f}, blur_s={blur_s:.4f}, blur_joint={blur_joint:.4f}"
)

In [ ]:
def split_pool_holdout_sinkhorn(
    df_region: pd.DataFrame,
    monthly_cols: list[str],
    static_cols: list[str],
    scaler_m,
    scaler_s,
    blur_joint: float,
    glacier_col: str = "GLACIER",
    id_col: str = "ID",
    elev_diff_col: str = "ELEVATION_DIFFERENCE",
    pool_frac: float = 0.2,
    seed: int = 0,
    n_restarts: int = 50,
    frac_tol: float = 0.05,
    min_pool_glaciers: int = 3,
) -> dict:
    """
    Split a target region into a fine-tuning pool and a holdout set.

    Distances are computed at measurement-row level using the true joint
    (climate + topo stacked) Sinkhorn divergence, consistent with
    D_sinkhorn_joint in compute_domain_shift.

    The split is optimized so that:
      (1) Sinkhorn(pool → holdout) is small  — pool covers the holdout distribution
      (2) |Sinkhorn(pool → all) - Sinkhorn(holdout → all)| is small — balanced shift

    Parameters
    ----------
    df_region : pd.DataFrame
        All rows for the target region.
    monthly_cols : list[str]
        Monthly feature column names.
    static_cols : list[str]
        Static feature column names.
    scaler_m :
        Global monthly scaler fitted via build_global_scalers_multi_source.
    scaler_s :
        Global static scaler fitted via build_global_scalers_multi_source.
    blur_joint : float
        Sinkhorn blur for the joint space — pass blur_joint from 1.7.
    glacier_col : str
        Column identifying glaciers.
    id_col : str
        Column identifying individual measurements.
    elev_diff_col : str
        Name of the elevation difference column (averaged per ID, not .first()).
    pool_frac : float
        Target fraction of measurements to use as fine-tuning pool.
    seed : int
        Base random seed.
    n_restarts : int
        Number of random orderings to try; best split by score is kept.
    frac_tol : float
        Splits where |achieved_frac - pool_frac| > frac_tol are skipped.
    min_pool_glaciers : int
        Minimum number of glaciers required in the pool.

    Returns
    -------
    dict with keys:
        pool_glaciers, holdout_glaciers,
        n_meas_pool, n_meas_holdout,
        actual_pool_frac,
        sinkhorn_pool_vs_holdout,
        sinkhorn_pool_vs_region,
        sinkhorn_holdout_vs_region,
        blur_joint, best_seed, best_score
    """
    # ------------------------------------------------------------------
    # 1. Glacier inventory — names and measurement counts
    # ------------------------------------------------------------------
    glaciers = df_region[glacier_col].unique().tolist()
    n_glaciers = len(glaciers)
    glacier_to_idx = {g: i for i, g in enumerate(glaciers)}

    n_meas = np.array([(df_region[glacier_col] == g).sum() for g in glaciers],
                      dtype=int)

    n_total_meas = int(n_meas.sum())
    target_pool_meas = int(round(pool_frac * n_total_meas))

    print(f"  Total measurements : {n_total_meas}")
    print(f"  Target pool meas   : {target_pool_meas} ({pool_frac:.0%})")
    print(f"  Total glaciers     : {n_glaciers}")
    print(f"  blur_joint         : {blur_joint:.4f}")

    # ------------------------------------------------------------------
    # 2. Precompute row-level joint features for the whole region
    #    (consistent with compute_domain_shift's D_sinkhorn_joint)
    # ------------------------------------------------------------------
    pure_static = [c for c in static_cols if c != elev_diff_col]

    def _stake_topo(df):
        parts = [df.groupby(id_col)[pure_static].first()]
        if elev_diff_col in static_cols:
            parts.append(df.groupby(id_col)[[elev_diff_col]].mean())
        return pd.concat(parts, axis=1)[static_cols].to_numpy(dtype=np.float64)

    Xm_all = scaler_m.transform(
        df_region[monthly_cols].to_numpy(dtype=np.float64))
    Xs_id = scaler_s.transform(_stake_topo(df_region))
    id_codes = pd.Categorical(df_region[id_col]).codes
    topo_per_row = Xs_id[id_codes]
    X_joint_all = np.hstack([Xm_all, topo_per_row]).astype(np.float32)

    # row-to-glacier mapping for fast subsetting
    glacier_codes = np.array(
        [glacier_to_idx[g] for g in df_region[glacier_col]], dtype=int)

    loss_fn = SamplesLoss(
        loss="sinkhorn",
        p=2,
        blur=blur_joint,
        scaling=0.9,
        debias=True,
        backend="tensorized",
    )

    def _sinkhorn(mask_a, mask_b, max_samples=2000):
        Xa = X_joint_all[mask_a]
        Xb = X_joint_all[mask_b]
        if len(Xa) < 2 or len(Xb) < 2:
            return 0.0
        if len(Xa) > max_samples:
            Xa = Xa[np.random.choice(len(Xa), max_samples, replace=False)]
        if len(Xb) > max_samples:
            Xb = Xb[np.random.choice(len(Xb), max_samples, replace=False)]
        ta = torch.as_tensor(Xa, dtype=torch.float32)
        tb = torch.as_tensor(Xb, dtype=torch.float32)
        wa = torch.ones(len(ta)) / len(ta)
        wb = torch.ones(len(tb)) / len(tb)
        with torch.no_grad():
            return float(loss_fn(wa, ta, wb, tb).item())

    def _mask_for(glacier_idxs):
        return np.isin(glacier_codes, list(glacier_idxs))

    all_mask = np.ones(len(X_joint_all), dtype=bool)

    # ------------------------------------------------------------------
    # 3. Greedy split with n_restarts random orderings, keep best score
    # ------------------------------------------------------------------
    best_score = np.inf
    best_result = None

    for restart in range(n_restarts):
        rng = np.random.default_rng(seed + restart)
        order = np.arange(n_glaciers)
        rng.shuffle(order)

        pool_idxs = set()
        holdout_idxs = set()
        pool_meas_count = 0

        for glacier_idx in order:
            gl_meas = int(n_meas[glacier_idx])
            pool_full = pool_meas_count + gl_meas > target_pool_meas * (
                1 + frac_tol)

            if pool_full:
                holdout_idxs.add(glacier_idx)
                continue

            trial_pool = pool_idxs | {glacier_idx}
            trial_holdout = holdout_idxs | {glacier_idx}

            holdout_mask = _mask_for(
                holdout_idxs) if holdout_idxs else all_mask
            pool_mask = _mask_for(pool_idxs) if pool_idxs else all_mask

            score_if_pool = (_sinkhorn(_mask_for(trial_pool), holdout_mask) +
                             abs(
                                 _sinkhorn(_mask_for(trial_pool), all_mask) -
                                 _sinkhorn(holdout_mask, all_mask)))
            score_if_holdout = (
                _sinkhorn(pool_mask, _mask_for(trial_holdout)) + abs(
                    _sinkhorn(pool_mask, all_mask) -
                    _sinkhorn(_mask_for(trial_holdout), all_mask)))

            if score_if_pool <= score_if_holdout:
                pool_idxs.add(glacier_idx)
                pool_meas_count += gl_meas
            else:
                holdout_idxs.add(glacier_idx)

        achieved_frac = pool_meas_count / n_total_meas
        if abs(achieved_frac - pool_frac) > frac_tol:
            continue
        if len(pool_idxs) < min_pool_glaciers:
            continue

        pool_mask = _mask_for(pool_idxs)
        holdout_mask = _mask_for(holdout_idxs)

        sk_pool_holdout = _sinkhorn(pool_mask, holdout_mask)
        sk_pool_region = _sinkhorn(pool_mask, all_mask)
        sk_holdout_region = _sinkhorn(holdout_mask, all_mask)

        score = sk_pool_holdout + abs(sk_pool_region - sk_holdout_region)

        print(f"  restart {restart:3d} | frac={achieved_frac:.2f} | "
              f"sk(pool↔holdout)={sk_pool_holdout:.4f} | "
              f"balance={abs(sk_pool_region - sk_holdout_region):.4f} | "
              f"score={score:.4f}")

        if score < best_score:
            best_score = score
            best_result = {
                "pool_idxs": pool_idxs,
                "holdout_idxs": holdout_idxs,
                "pool_meas_count": pool_meas_count,
                "achieved_frac": achieved_frac,
                "sk_pool_holdout": sk_pool_holdout,
                "sk_pool_region": sk_pool_region,
                "sk_holdout_region": sk_holdout_region,
                "best_seed": seed + restart,
            }

    if best_result is None:
        raise RuntimeError(
            f"No valid split found within frac_tol={frac_tol} and "
            f"min_pool_glaciers={min_pool_glaciers} after {n_restarts} restarts. "
            f"Try increasing n_restarts or frac_tol.")

    r = best_result
    pool_glaciers = [glaciers[i] for i in r["pool_idxs"]]
    holdout_glaciers = [glaciers[i] for i in r["holdout_idxs"]]

    print(f"\n  Best split (seed={r['best_seed']}):")
    print(
        f"  Pool    : {len(pool_glaciers)} glaciers, {r['pool_meas_count']} meas ({r['achieved_frac']:.1%})"
    )
    print(
        f"  Holdout : {len(holdout_glaciers)} glaciers, {n_total_meas - r['pool_meas_count']} meas ({1 - r['achieved_frac']:.1%})"
    )
    print(f"  Sinkhorn(pool → holdout)  : {r['sk_pool_holdout']:.4f}")
    print(f"  Sinkhorn(pool → region)   : {r['sk_pool_region']:.4f}")
    print(f"  Sinkhorn(holdout → region): {r['sk_holdout_region']:.4f}")
    print(f"  Score                     : {best_score:.4f}")

    return {
        "pool_glaciers": pool_glaciers,
        "holdout_glaciers": holdout_glaciers,
        "n_meas_pool": int(r["pool_meas_count"]),
        "n_meas_holdout": int(n_total_meas - r["pool_meas_count"]),
        "actual_pool_frac": float(r["achieved_frac"]),
        "sinkhorn_pool_vs_holdout": float(r["sk_pool_holdout"]),
        "sinkhorn_pool_vs_region": float(r["sk_pool_region"]),
        "sinkhorn_holdout_vs_region": float(r["sk_holdout_region"]),
        "blur_joint": float(blur_joint),
        "best_seed": int(r["best_seed"]),
        "best_score": float(best_score),
    }

#### FT Pool & test set:

In [ ]:
ISL_SPLIT_PATH = BASE_DIR / "isl_test_pool_split.json"
RECOMPUTE_ISL_SPLIT = False

df_isl_all = res_xreg_by_source["CH"]["df_test"]
df_isl_all = df_isl_all[df_isl_all["SOURCE_CODE"] == "ISL"].copy()

if not RECOMPUTE_ISL_SPLIT and ISL_SPLIT_PATH.exists():
    with open(ISL_SPLIT_PATH) as f:
        isl_split = json.load(f)
    print(f"Loaded ISL split from {ISL_SPLIT_PATH}")
else:
    isl_split = split_pool_holdout_sinkhorn(
        df_region=df_isl_all,
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
        scaler_m=scaler_m,
        scaler_s=scaler_s,
        blur_joint=blur_joint,
        pool_frac=0.8,
        n_restarts=50,
        seed=cfg.seed,
    )
    with open(ISL_SPLIT_PATH, "w") as f:
        json.dump(isl_split, f, indent=2)
    print(f"Saved ISL split to {ISL_SPLIT_PATH}")

ISL_FT_POOL = isl_split["pool_glaciers"]
ISL_TEST_SET = isl_split["holdout_glaciers"]

df_isl_pool = df_isl_all[df_isl_all["GLACIER"].isin(ISL_FT_POOL)]
df_isl_test = df_isl_all[df_isl_all["GLACIER"].isin(ISL_TEST_SET)]

print(
    f"FT pool : {len(ISL_FT_POOL)} glaciers | {len(df_isl_pool)} measurements")
print(
    f"Test set: {len(ISL_TEST_SET)} glaciers | {len(df_isl_test)} measurements"
)

# ── Sinkhorn distances from the split ────────────────────────────────────
print("\nSinkhorn distances (from best split):")
print(f"  Sk(pool  → test set) : {isl_split['sinkhorn_pool_vs_holdout']:.4f}  "
      f"← how well pool covers test distribution")
print(f"  Sk(pool  → all ISL)  : {isl_split['sinkhorn_pool_vs_region']:.4f}  "
      f"← pool representativeness of full region")
print(
    f"  Sk(test  → all ISL)  : {isl_split['sinkhorn_holdout_vs_region']:.4f}  "
    f"← test set representativeness of full region")
print(
    f"  Balance |pool-test|  : "
    f"{abs(isl_split['sinkhorn_pool_vs_region'] - isl_split['sinkhorn_holdout_vs_region']):.4f}  "
    f"← should be small for a fair split")
print(f"  Best score           : {isl_split['best_score']:.4f}")
print(f"  Best seed            : {isl_split['best_seed']}")
print(f"  Actual pool fraction : {isl_split['actual_pool_frac']:.1%}")

In [ ]:
from matplotlib.ticker import FuncFormatter
import scipy.stats as scipy_stats


def format_axis_ticks(ax, label_size=8):
    """Format tick labels to avoid huge 1e6/1e7 offset labels."""
    # check if scientific notation offset is being used
    ax.xaxis.get_major_formatter().set_useOffset(False)
    try:
        # scale large numbers to readable units
        xmax = abs(ax.get_xlim()[1])
        if xmax > 1e6:
            scale = 1e6
            ax.xaxis.set_major_formatter(
                FuncFormatter(lambda x, _: f"{x/scale:.1f}"))
            ax.set_xlabel(f"(×10⁶)", label_size=label_size, labelpad=1)
        elif xmax > 1e4:
            scale = 1e3
            ax.xaxis.set_major_formatter(
                FuncFormatter(lambda x, _: f"{x/scale:.0f}"))
            ax.set_xlabel(f"(×10³)", label_size=label_size, labelpad=1)
    except Exception:
        pass


def plot_kde_pair(glaciers_to_plot, selected_cols, save_prefix):
    """KDE panels for a pair of glaciers."""
    ncols = 3
    nrows = int(np.ceil(len(selected_cols) / ncols))
    w, h = nature_figsize(cols=1, height_mm=160)
    fig, axes = plt.subplots(nrows, ncols, figsize=(w * 2, h), squeeze=False)

    legend_handles = []

    for idx, col in enumerate(selected_cols):
        ax = axes[idx // ncols][idx % ncols]

        all_vals = pd.concat([
            cfg_gl["df"][col].dropna() for cfg_gl in glaciers_to_plot.values()
        ])
        x_grid = np.linspace(float(all_vals.min()), float(all_vals.max()), 500)

        for label, cfg_gl in glaciers_to_plot.items():
            vals = cfg_gl["df"][col].dropna().values
            if len(vals) < 10:
                continue
            kde = scipy_stats.gaussian_kde(vals, bw_method=0.3)
            y = kde(x_grid)
            y = y / y.max()
            line, = ax.plot(x_grid,
                            y,
                            color=cfg_gl["color"],
                            linewidth=0.8,
                            label=label)
            ax.fill_between(x_grid, y, alpha=0.08, color=cfg_gl["color"])

            # collect handles only from first panel to avoid duplicates
            if idx == 0:
                legend_handles.append(line)

        ax.set_title(col, fontsize=8)
        ax.set_ylim(0, 1.15)
        ax.set_xlabel("")
        ax.tick_params(labelsize=8, width=0.4, length=2, direction="in")
        ax.spines[["top", "right", "left", "bottom"]].set_visible(True)
        for spine in ax.spines.values():
            spine.set_linewidth(0.4)
        ax.grid(axis="x", color="#e0e0e0", linewidth=0.3)
        ax.set_axisbelow(True)
        format_axis_ticks(ax, label_size=6)

    # ── place legend in first empty axis if one exists, else above figure ──
    empty_axes = [
        axes[idx // ncols][idx % ncols]
        for idx in range(len(selected_cols), nrows * ncols)
    ]

    if empty_axes:
        leg_ax = empty_axes[0]
        leg_ax.axis("off")
        leg_ax.legend(
            handles=legend_handles,
            loc="center",
            fontsize=8,
            frameon=False,
        )
        # turn off remaining empty axes
        for ax in empty_axes[1:]:
            ax.axis("off")
    else:
        # no empty panels — place legend above the figure
        fig.legend(
            handles=legend_handles,
            loc="upper center",
            bbox_to_anchor=(0.5, 1.02),
            ncol=len(glaciers_to_plot),
            fontsize=8,
            frameon=False,
        )

    plt.tight_layout(h_pad=3.0)
    plt.savefig(f"figures/paperTF/{save_prefix}_kde.pdf", bbox_inches="tight")
    plt.savefig(f"figures/paperTF/{save_prefix}_kde.png",
                dpi=300,
                bbox_inches="tight")
    plt.show()

In [ ]:
selected_cols = MONTHLY_COLS + [
    c for c in STATIC_COLS if c != "ELEVATION_DIFFERENCE"
]

# SELECTED_COLS = [
#     't2m',
#     'tp',
#     'ssrd',
#     'ELEVATION_DIFFERENCE',
#     'aspect',
#     'slope',
# ]

glaciers_to_plot = {
    f"Full ISL ({df_isl_all['GLACIER'].nunique()} glaciers)": {
        "df": df_isl_all,
        "color": "#4dac26",  # green
    },
    f"FT pool ({len(ISL_FT_POOL)} glaciers)": {
        "df": df_isl_pool,
        "color": "#2166ac",  # blue
    },
    f"Test set ({len(ISL_TEST_SET)} glaciers)": {
        "df": df_isl_test,
        "color": "#b2182b",  # red
    },
}

plot_kde_pair(
    glaciers_to_plot=glaciers_to_plot,
    selected_cols=selected_cols,
    save_prefix="isl_pool_vs_test",
)

#### FT Pool subsets:

In [ ]:
# ── Helper: run N_RESTARTS, return top-K results ──────────────────────────
def _run_sinkhorn_top_k_clustered(
    df_pool, monthly_cols, static_cols, scaler_m, scaler_s,
    blur_joint, pool_frac, mode,
    topo_extra_cols=None,
    seed=0, n_restarts=50, frac_tol=0.05, min_pool_ids=5,
    top_k=10, id_col="ID", elev_diff_col="ELEVATION_DIFFERENCE",
    n_clusters=300,       # number of ID clusters for greedy loop
    max_samples=500,      # max rows per Sinkhorn call
    patience=15,          # early stopping: max restarts without improvement
):
    """
    Same as _run_sinkhorn_top_k but with two speedups:
      1. IDs are clustered into n_clusters representatives — greedy loop
         runs on clusters, not individual IDs (~10-20x fewer decisions)
      2. max_samples reduced for faster Sinkhorn calls
      3. Early stopping when no improvement after `patience` restarts
    
    Selecting a cluster = selecting all IDs in that cluster (they move together).
    """
    from sklearn.cluster import MiniBatchKMeans

    topo_extra_cols = topo_extra_cols or []

    # ------------------------------------------------------------------
    # 1. ID inventory
    # ------------------------------------------------------------------
    ids      = df_pool[id_col].unique().tolist()
    n_ids    = len(ids)
    id_to_idx = {id_: i for i, id_ in enumerate(ids)}

    n_meas_per_id = np.array(
        [(df_pool[id_col] == id_).sum() for id_ in ids], dtype=int
    )
    n_total_meas = int(n_meas_per_id.sum())
    target_meas  = int(round(pool_frac * n_total_meas))

    print(f"  Mode               : {mode.upper()}")
    print(f"  Total IDs          : {n_ids}")
    print(f"  Total measurements : {n_total_meas}")
    print(f"  Target selected    : {target_meas} ({pool_frac:.0%})")
    print(f"  n_clusters         : {n_clusters}")
    print(f"  max_samples        : {max_samples}")
    print(f"  patience           : {patience}")

    # ------------------------------------------------------------------
    # 2. Feature matrix (identical to original)
    # ------------------------------------------------------------------
    pure_static = [c for c in static_cols if c != elev_diff_col]

    if mode == "joint":
        Xm_all = scaler_m.transform(
            df_pool[monthly_cols].to_numpy(dtype=np.float64)
        )
        static_per_id = df_pool.groupby(id_col)[pure_static].first()
        if elev_diff_col in static_cols:
            static_per_id[elev_diff_col] = df_pool.groupby(id_col)[elev_diff_col].mean()
        Xs_id = scaler_s.transform(
            static_per_id[static_cols].to_numpy(dtype=np.float64)
        )
        id_codes     = np.array([id_to_idx[i] for i in df_pool[id_col]], dtype=int)
        topo_per_row = Xs_id[id_codes]
        X = np.hstack([Xm_all, topo_per_row]).astype(np.float32)
        blur = blur_joint

    else:  # topo
        static_per_id = df_pool.groupby(id_col)[pure_static].first()
        if elev_diff_col in static_cols:
            static_per_id[elev_diff_col] = df_pool.groupby(id_col)[elev_diff_col].mean()
        Xs_static = scaler_s.transform(
            static_per_id[static_cols].to_numpy(dtype=np.float64)
        )
        extra_parts = []
        for col in topo_extra_cols:
            col_idx  = monthly_cols.index(col)
            col_vals = (
                df_pool.groupby(id_col)[col].mean()
                .loc[ids]
                .to_numpy(dtype=np.float64)
                .reshape(-1, 1)
            )
            extra_parts.append(
                (col_vals - scaler_m.mean_[col_idx]) / scaler_m.scale_[col_idx]
            )
        Xs_id = np.hstack([Xs_static] + extra_parts).astype(np.float32) \
                if extra_parts else Xs_static.astype(np.float32)
        id_codes = np.array([id_to_idx[i] for i in df_pool[id_col]], dtype=int)
        X = Xs_id[id_codes]
        all_topo_cols = static_cols + topo_extra_cols
        n_topo = len(all_topo_cols)
        n_joint = len(monthly_cols) + len(static_cols)
        blur = blur_joint * np.sqrt(n_topo / n_joint)
        print(f"  Topo features      : {all_topo_cols}")
        print(f"  blur_topo          : {blur:.4f}")

    # ------------------------------------------------------------------
    # 3. Cluster IDs into n_clusters representatives
    #    Use per-ID mean feature vector as the clustering input
    # ------------------------------------------------------------------
    print(f"\n  Clustering {n_ids} IDs into {n_clusters} clusters...")

    # One mean feature vector per ID
    id_mean_features = np.zeros((n_ids, X.shape[1]), dtype=np.float32)
    for idx in range(n_ids):
        mask = id_codes == idx
        if mask.any():
            id_mean_features[idx] = X[mask].mean(axis=0)

    n_clusters_actual = min(n_clusters, n_ids)
    kmeans = MiniBatchKMeans(
        n_clusters=n_clusters_actual,
        random_state=seed,
        n_init=3,
        max_iter=100,
    )
    cluster_labels = kmeans.fit_predict(id_mean_features)  # shape: (n_ids,)

    # Build cluster → ID index mapping and cluster measurement counts
    clusters      = list(range(n_clusters_actual))
    cluster_to_id_idxs = {c: [] for c in clusters}
    for id_idx, c in enumerate(cluster_labels):
        cluster_to_id_idxs[c].append(id_idx)

    cluster_meas = np.array([
        sum(n_meas_per_id[i] for i in cluster_to_id_idxs[c])
        for c in clusters
    ], dtype=int)

    print(f"  Cluster sizes (meas): min={cluster_meas.min()} "
          f"max={cluster_meas.max()} mean={cluster_meas.mean():.0f}")

    # ------------------------------------------------------------------
    # 4. Sinkhorn helper — operates on row-level X, mask by id_codes
    # ------------------------------------------------------------------
    loss_fn = SamplesLoss(
    loss="sinkhorn", p=2, blur=blur,
    scaling=0.9, debias=True,
    backend="tensorized",  # ← back to tensorized, online needs KeOps CUDA
    )
    def _mask_for_id_idxs(id_idxs):
        return np.isin(id_codes, list(id_idxs))

    def _sinkhorn(mask_a, mask_b):
        Xa = X[mask_a]
        Xb = X[mask_b]
        if len(Xa) < 2 or len(Xb) < 2:
            return 0.0
        if len(Xa) > max_samples:
            Xa = Xa[np.random.choice(len(Xa), max_samples, replace=False)]
        if len(Xb) > max_samples:
            Xb = Xb[np.random.choice(len(Xb), max_samples, replace=False)]
        ta = torch.as_tensor(Xa, dtype=torch.float32)
        tb = torch.as_tensor(Xb, dtype=torch.float32)
        wa = torch.ones(len(ta)) / len(ta)
        wb = torch.ones(len(tb)) / len(tb)
        with torch.no_grad():
            return float(loss_fn(wa, ta, wb, tb).item())

    def _mask_for_clusters(cluster_idxs):
        """Get row mask for all IDs belonging to a set of clusters."""
        id_idxs = [
            id_idx
            for c in cluster_idxs
            for id_idx in cluster_to_id_idxs[c]
        ]
        return _mask_for_id_idxs(id_idxs)

    all_mask = np.ones(len(X), dtype=bool)

    # ------------------------------------------------------------------
    # 5. Greedy restarts over CLUSTERS (not IDs)
    # ------------------------------------------------------------------
    all_results = []
    best_score  = np.inf
    no_improve  = 0

    for restart in range(n_restarts):
        rng   = np.random.default_rng(seed + restart)
        order = np.arange(n_clusters_actual)
        rng.shuffle(order)

        sel_clusters   = set()
        unsel_clusters = set()
        sel_meas       = 0

        for c_idx in order:
            c_meas    = int(cluster_meas[c_idx])
            if sel_meas + c_meas > target_meas * (1 + frac_tol):
                unsel_clusters.add(c_idx)
                continue

            trial_sel   = sel_clusters   | {c_idx}
            trial_unsel = unsel_clusters | {c_idx}

            unsel_mask = _mask_for_clusters(unsel_clusters) if unsel_clusters else all_mask
            sel_mask   = _mask_for_clusters(sel_clusters)   if sel_clusters   else all_mask

            score_if_sel = (
                _sinkhorn(_mask_for_clusters(trial_sel), unsel_mask) +
                abs(_sinkhorn(_mask_for_clusters(trial_sel), all_mask) -
                    _sinkhorn(unsel_mask, all_mask))
            )
            score_if_unsel = (
                _sinkhorn(sel_mask, _mask_for_clusters(trial_unsel)) +
                abs(_sinkhorn(sel_mask, all_mask) -
                    _sinkhorn(_mask_for_clusters(trial_unsel), all_mask))
            )

            if score_if_sel <= score_if_unsel:
                sel_clusters.add(c_idx)
                sel_meas += c_meas
            else:
                unsel_clusters.add(c_idx)

        achieved_frac = sel_meas / n_total_meas
        if abs(achieved_frac - pool_frac) > frac_tol:
            no_improve += 1
            if no_improve >= patience:
                print(f"  Early stopping at restart {restart} (patience={patience})")
                break
            continue

        # Expand clusters back to individual IDs
        sel_id_idxs = [
            id_idx
            for c in sel_clusters
            for id_idx in cluster_to_id_idxs[c]
        ]
        sel_mask   = _mask_for_id_idxs(sel_id_idxs)
        unsel_id_idxs = [
            id_idx
            for c in unsel_clusters
            for id_idx in cluster_to_id_idxs[c]
        ]
        unsel_mask = _mask_for_id_idxs(unsel_id_idxs) if unsel_id_idxs else ~sel_mask

        sk_sel_unsel  = _sinkhorn(sel_mask, unsel_mask)
        sk_sel_pool   = _sinkhorn(sel_mask, all_mask)
        sk_unsel_pool = _sinkhorn(unsel_mask, all_mask)
        score = sk_sel_unsel + abs(sk_sel_pool - sk_unsel_pool)

        print(f"  restart {restart:3d} | frac={achieved_frac:.3f} | "
              f"clusters={len(sel_clusters)} | "
              f"sk(sel↔unsel)={sk_sel_unsel:.4f} | "
              f"balance={abs(sk_sel_pool-sk_unsel_pool):.4f} | "
              f"score={score:.4f}")

        all_results.append((score, {
            "sel_id_idxs":   sel_id_idxs,
            "unsel_id_idxs": unsel_id_idxs,
            "sel_clusters":  sel_clusters,
            "sel_meas":      sel_meas,
            "achieved_frac": achieved_frac,
            "sk_sel_unsel":  sk_sel_unsel,
            "sk_sel_pool":   sk_sel_pool,
            "sk_unsel_pool": sk_unsel_pool,
            "restart":       restart,
            "seed":          seed + restart,
        }))

        if score < best_score - 1e-4:
            best_score = score
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"  Early stopping at restart {restart} (patience={patience})")
                break

    if not all_results:
        raise RuntimeError(
            f"No valid split found after {n_restarts} restarts. "
            f"Try increasing n_restarts, frac_tol, or reducing n_clusters."
        )

    all_results.sort(key=lambda x: x[0])
    top_k_results = all_results[:top_k]

    print(f"\n  Valid restarts : {len(all_results)}")
    print(f"  Score range    : [{all_results[0][0]:.4f} – {all_results[-1][0]:.4f}]")
    print(f"  Top-{top_k} scores : {[round(r[0], 4) for r in top_k_results]}")

    # ------------------------------------------------------------------
    # 6. Format output — expand clusters back to IDs and glaciers
    # ------------------------------------------------------------------
    top_k_splits = []
    for rank, (score, r) in enumerate(top_k_results):
        sel_ids      = [ids[i] for i in r["sel_id_idxs"]]
        unsel_ids    = [ids[i] for i in r["unsel_id_idxs"]]
        sel_glaciers = df_pool[df_pool[id_col].isin(sel_ids)]["GLACIER"].unique().tolist()

        top_k_splits.append({
            "rank":              rank,
            "score":             float(score),
            "selected_ids":      sel_ids,
            "unselected_ids":    unsel_ids,
            "selected_glaciers": sel_glaciers,
            "n_ids":             len(sel_ids),
            "n_glaciers":        len(sel_glaciers),
            "n_meas":            int(r["sel_meas"]),
            "actual_frac":       float(r["achieved_frac"]),
            "sk_sel_unsel":      float(r["sk_sel_unsel"]),
            "sk_sel_pool":       float(r["sk_sel_pool"]),
            "sk_unsel_pool":     float(r["sk_unsel_pool"]),
            "n_clusters_used":   int(len(r["sel_clusters"])),
            "restart":           int(r["restart"]),
            "seed":              int(r["seed"]),
        })

    return top_k_splits

In [ ]:
# ── Step B: Generate all FT subsets (informed joint + topo + random) ─────

import json
from pathlib import Path

FRACS = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.5]
N_SEEDS      = 10
N_TOP        = 10   # number of best restarts to save per fraction
N_RESTARTS   = 50
N_TOTAL_ISL  = len(df_isl_all)

SUBSETS_PATH     = BASE_DIR / "isl_ft_subsets_combined.json"
RECOMPUTE_JOINT  = True
RECOMPUTE_TOPO   = True
RECOMPUTE_RANDOM = True

# ── Load or init ──────────────────────────────────────────────────────────
if SUBSETS_PATH.exists() and not (RECOMPUTE_JOINT or RECOMPUTE_TOPO or RECOMPUTE_RANDOM):
    with open(SUBSETS_PATH) as f:
        ft_subsets = json.load(f)
    print(f"Loaded FT subsets from {SUBSETS_PATH}")
else:
    ft_subsets = {
        "informed_joint": {},
        "informed_topo":  {},
        "random":         {},
    }

# ── Informed joint splits ─────────────────────────────────────────────────
if RECOMPUTE_JOINT:
    ft_subsets["informed_joint"] = {}
    for frac in FRACS:
        pct = int(frac * 100)
        pool_frac_relative = int(round(frac * N_TOTAL_ISL)) / len(df_isl_pool)
        print(f"\n{'='*55}")
        print(f"Informed JOINT @ {pct}%  (pool_frac={pool_frac_relative:.3f})")
        top_k = _run_sinkhorn_top_k_clustered(
            df_pool=df_isl_pool,
            monthly_cols=MONTHLY_COLS, static_cols=STATIC_COLS,
            scaler_m=scaler_m, scaler_s=scaler_s,
            blur_joint=blur_joint, pool_frac=pool_frac_relative,
            mode="joint", seed=cfg.seed,
            n_restarts=N_RESTARTS, top_k=N_TOP,
            n_clusters=300,    # tune this — fewer = faster, more = finer-grained
            max_samples=500,
            patience=15,
        )
        ft_subsets["informed_joint"][str(pct)] = top_k
        best = top_k[0]
        print(f"  Best → {best['n_ids']} IDs | {best['n_glaciers']} glaciers | "
              f"{best['n_meas']} meas ({best['actual_frac']:.1%}) | score={best['score']:.4f}")

# ── Informed topo splits ──────────────────────────────────────────────────
if RECOMPUTE_TOPO:
    ft_subsets["informed_topo"] = {}
    for frac in FRACS:
        pct = int(frac * 100)
        pool_frac_relative = int(round(frac * N_TOTAL_ISL)) / len(df_isl_pool)
        print(f"\n{'='*55}")
        print(f"Informed TOPO @ {pct}%  (pool_frac={pool_frac_relative:.3f})")
        top_k = _run_sinkhorn_top_k_clustered(
            df_pool=df_isl_pool,
            monthly_cols=MONTHLY_COLS, static_cols=STATIC_COLS,
            scaler_m=scaler_m, scaler_s=scaler_s,
            blur_joint=blur_joint, pool_frac=pool_frac_relative,
            mode="topo", topo_extra_cols=["ELEVATION_DIFFERENCE"],
            seed=cfg.seed, n_restarts=N_RESTARTS, top_k=N_TOP,
            n_clusters=300,    # tune this — fewer = faster, more = finer-grained
            max_samples=500,
            patience=15,
        )
        ft_subsets["informed_topo"][str(pct)] = top_k
        best = top_k[0]
        print(f"  Best → {best['n_ids']} IDs | {best['n_glaciers']} glaciers | "
              f"{best['n_meas']} meas ({best['actual_frac']:.1%}) | score={best['score']:.4f}")

# ── Random splits ─────────────────────────────────────────────────────────
if RECOMPUTE_RANDOM or "random" not in ft_subsets:
    ft_subsets["random"] = {}
    rng = np.random.default_rng(cfg.seed)

    for frac in FRACS:
        pct = int(frac * 100)
        target_meas = int(round(frac * N_TOTAL_ISL))
        ft_subsets["random"][str(pct)] = []

        # cap random at the best informed_joint measurement count for fairness
        informed_n_meas = ft_subsets["informed_joint"][str(pct)][0]["n_meas"]
        g_counts = df_isl_pool.groupby("GLACIER").size().loc[ISL_FT_POOL]

        print(f"\nRandom @ {pct}%  (target {target_meas} meas, "
              f"capped at {informed_n_meas} meas)")

        for seed_idx in range(N_SEEDS):
            seed = int(rng.integers(0, 2**31))
            rng_local = np.random.default_rng(seed)

            shuffled = list(rng_local.permutation(ISL_FT_POOL))
            selected_glaciers, n_meas_selected = [], 0
            for g in shuffled:
                selected_glaciers.append(g)
                n_meas_selected += int(g_counts[g])
                if n_meas_selected >= target_meas:
                    break

            df_selected = df_isl_pool[df_isl_pool["GLACIER"].isin(selected_glaciers)]
            if n_meas_selected > informed_n_meas:
                all_ids = list(df_selected["ID"].unique())
                rng_local.shuffle(all_ids)
                kept_ids, n_kept = [], 0
                for id_ in all_ids:
                    id_meas = int((df_selected["ID"] == id_).sum())
                    if n_kept + id_meas <= informed_n_meas:
                        kept_ids.append(id_)
                        n_kept += id_meas
                selected_glaciers = list(
                    df_selected[df_selected["ID"].isin(kept_ids)]["GLACIER"].unique()
                )
                n_meas_selected = n_kept

            ft_subsets["random"][str(pct)].append({
                "seed_idx":    seed_idx,
                "seed":        seed,
                "glaciers":    selected_glaciers,
                "n_glaciers":  len(selected_glaciers),
                "n_meas":      n_meas_selected,
                "actual_frac": n_meas_selected / N_TOTAL_ISL,
            })

        actual_fracs = [s["actual_frac"] for s in ft_subsets["random"][str(pct)]]
        print(f"  → avg {sum(s['n_glaciers'] for s in ft_subsets['random'][str(pct)])/N_SEEDS:.1f} glaciers | "
              f"frac range [{min(actual_fracs):.1%} – {max(actual_fracs):.1%}]")

# ── Save ──────────────────────────────────────────────────────────────────
if RECOMPUTE_JOINT or RECOMPUTE_TOPO or RECOMPUTE_RANDOM:
    with open(SUBSETS_PATH, "w") as f:
        json.dump(ft_subsets, f, indent=2)
    print(f"\nSubsets saved to {SUBSETS_PATH}")

# ── Sanity check ──────────────────────────────────────────────────────────
print("\nSubset summary (best split = rank 0):")
print(f"{'':6} {'joint (best)':>22} {'topo (best)':>22} {'random (mean)':>25}")
print(f"{'frac':6} {'IDs':>6} {'glac':>5} {'meas':>7}  "
      f"{'IDs':>6} {'glac':>5} {'meas':>7}  "
      f"{'glac':>6} {'meas':>7} {'actual%':>8}")
for frac in FRACS:
    pct = str(int(frac * 100))
    j   = ft_subsets["informed_joint"][pct][0]   # best joint
    t   = ft_subsets["informed_topo"][pct][0]    # best topo
    rnd = ft_subsets["random"][pct]
    print(
        f"{int(pct):>4}%  "
        f"{j['n_ids']:>6}  {j['n_glaciers']:>4}  {j['n_meas']:>7}  "
        f"{t['n_ids']:>6}  {t['n_glaciers']:>4}  {t['n_meas']:>7}  "
        f"{sum(s['n_glaciers'] for s in rnd)/N_SEEDS:>6.1f}  "
        f"{sum(s['n_meas'] for s in rnd)/N_SEEDS:>7.0f}  "
        f"{sum(s['actual_frac'] for s in rnd)/N_SEEDS:>7.1%}"
    )

# ── Variability check: how many unique ID sets across top-K? ─────────────
print(f"\nTop-{N_TOP} variability (joint | topo):")
print(f"{'frac':6} {'unique joint sets':>18} {'unique topo sets':>17} "
      f"{'mean ID overlap (joint)':>24}")
for frac in FRACS:
    pct = str(int(frac * 100))
    joint_sets = [frozenset(r["selected_ids"]) for r in ft_subsets["informed_joint"][pct]]
    topo_sets  = [frozenset(r["selected_ids"]) for r in ft_subsets["informed_topo"][pct]]

    n_unique_j = len(set(joint_sets))
    n_unique_t = len(set(topo_sets))

    # mean pairwise Jaccard overlap between top-K joint sets
    pairs = [(joint_sets[i], joint_sets[j])
             for i in range(len(joint_sets))
             for j in range(i+1, len(joint_sets))]
    if pairs:
        mean_jaccard = np.mean([
            len(a & b) / len(a | b) for a, b in pairs
        ])
    else:
        mean_jaccard = 1.0

    print(f"{int(pct):>4}%  {n_unique_j:>18}  {n_unique_t:>17}  {mean_jaccard:>24.3f}")

In [ ]:
# pick any subset from ft_subsets
pct = "25"
strategy = "informed"  # or "random"
seed_idx = 0  # only relevant for random

if strategy == "informed":
    subset_glaciers = ft_subsets["informed"][pct]["glaciers"]
    label = f"Informed {pct}% ({len(subset_glaciers)} glaciers)"
else:
    subset_glaciers = ft_subsets["random"][pct][seed_idx]["glaciers"]
    label = f"Random {pct}% seed{seed_idx} ({len(subset_glaciers)} glaciers)"

df_subset = df_isl_pool[df_isl_pool["GLACIER"].isin(subset_glaciers)]

glaciers_to_plot = {
    f"Full ISL ({df_isl_all['GLACIER'].nunique()} glaciers)": {
        "df": df_isl_all,
        "color": "#4dac26",  # green
    },
    f"FT pool ({len(ISL_FT_POOL)} glaciers)": {
        "df": df_isl_pool,
        "color": "#2166ac",  # blue
    },
    label: {
        "df": df_subset,
        "color": "#b2182b" if strategy == "informed" else "#762a83",
    },
}

plot_kde_pair(
    glaciers_to_plot=glaciers_to_plot,
    selected_cols=selected_cols,
    save_prefix=
    f"isl_{strategy}_{pct}pct{'_seed'+str(seed_idx) if strategy == 'random' else ''}",
)

### Plot FT/Hold-out glaciers:

##### Iceland:

In [ ]:
ft_glaciers_by_split = {
    "exp_pool": ISL_FT_POOL,
}
holdout_glaciers_by_split = {
    "exp_pool": ISL_TEST_SET,
}

data_ISL, glacier_outline_rgi, glacier_info_by_split = build_region_glacier_info_for_splits(
    cfg,
    rgi_region_id="06",
    outline_shp_path=cfg.dataPath +
    "RGI_v6/RGI_06_Iceland/06_rgi60_Iceland.shp",
    ft_glaciers_by_split=ft_glaciers_by_split,
    split_names=("exp_pool", ),
    ft_label_col="FT/Hold-out glacier",
    holdout_glaciers_by_split=holdout_glaciers_by_split,
)

glacier_df_ISL_exp = glacier_info_by_split["exp_pool"]

train_color = get_cmap_hex(cm.batlow, 10)[0]
palette = {"Hold-out": train_color, "FT": "#b2182b"}

fig, ax, glacier_info_plot, scaled_size_fn = plot_glacier_measurements_map(
    glacier_info=glacier_df_ISL_exp,
    glacier_outline_rgi=glacier_outline_rgi,
    title="Glacier PMB location Iceland — experiment pool/test split",
    extent=(-25, -11, 62, 68),
    sizes=(100, 1500),
    size_legend_values=(30, 100, 1000),
    palette=palette,
    cmap_for_train=cm.batlow,
    split_col="FT/Hold-out glacier",
)

## LSTM model
### LSTM datasets:

In [ ]:
# ── Step C: Build TL assets for CH→ISL experiment ────────────────────────
# Structure:
#   - informed_joint: top-K splits × 8 fracs  (ID-level filtered)
#   - informed_topo:  top-K splits × 8 fracs  (ID-level filtered)
#   - random:         N_SEEDS splits × 8 fracs (glacier-level)

# ── Random: single call, glacier-level filtering works as before ──────────
FT_GLACIERS_RANDOM = {"ISL": {}}
for frac in FRACS:
    pct = int(frac * 100)
    for s in ft_subsets["random"][str(pct)]:
        key = f"ISL_random_{pct}pct_seed{s['seed_idx']}"
        FT_GLACIERS_RANDOM["ISL"][key] = s["glaciers"]

print(f"Random subsets: {len(FT_GLACIERS_RANDOM['ISL'])}")

tl_assets_random = build_transfer_learning_assets(
    cfg=cfg,
    res_xreg=res_xreg_by_source["CH"],
    FT_GLACIERS=FT_GLACIERS_RANDOM,
    MONTHLY_COLS=MONTHLY_COLS,
    STATIC_COLS=STATIC_COLS,
    cache_dir=BASE_DIR / "LSTM_cache_TL" / "CH" / "exp_ch_isl_random",
    force_recompute=False,
    src_code="CH",
    region_groups={},
)
print(f"Random TL assets: {len(tl_assets_random)}")

# ── Informed: ID-level filtering — one call per (strategy, frac, rank) ───
# build_transfer_learning_assets only supports glacier-level filtering via
# make_res_transfer_learning, so we pre-filter df_test/df_test_aug to the
# selected IDs before calling it.

def _build_informed_assets(strategy_key, cache_subdir, force_recompute=False):
    """
    Build TL assets for all fracs × top-K splits of one informed strategy.
    Returns a dict of assets keyed by experiment key.
    """
    assets = {}
    for frac in FRACS:
        pct = int(frac * 100)
        top_k_splits = ft_subsets[strategy_key][str(pct)]

        for rank, split in enumerate(top_k_splits):
            selected_ids      = split["selected_ids"]
            selected_glaciers = split["selected_glaciers"]
            exp_label         = f"ISL_{strategy_key}_{pct}pct_rank{rank}"

            print(f"  Building: {exp_label} | "
                  f"{len(selected_ids)} IDs | "
                  f"{len(selected_glaciers)} glaciers | "
                  f"score={split['score']:.4f}")

            # Pre-filter res_xreg so df_test/df_test_aug only contain
            # the selected IDs — this is how we achieve ID-level FT filtering
            res_ch = res_xreg_by_source["CH"]
            res_xreg_filtered = {
                **res_ch,
                "df_test": res_ch["df_test"][
                    res_ch["df_test"]["ID"].isin(selected_ids)
                ].copy(),
                "df_test_aug": res_ch["df_test_aug"][
                    res_ch["df_test_aug"]["ID"].isin(selected_ids)
                ].copy(),
            }

            assets_this = build_transfer_learning_assets(
                cfg=cfg,
                res_xreg=res_xreg_filtered,
                FT_GLACIERS={"ISL": {exp_label: selected_glaciers}},
                MONTHLY_COLS=MONTHLY_COLS,
                STATIC_COLS=STATIC_COLS,
                cache_dir=BASE_DIR / "LSTM_cache_TL" / "CH" / cache_subdir / f"{pct}pct_rank{rank}",
                force_recompute=force_recompute,
                src_code="CH",
                region_groups={},
            )
            assets.update(assets_this)

    return assets

print(f"\nBuilding informed_joint assets "
      f"({len(FRACS)} fracs × {N_TOP} ranks = {len(FRACS)*N_TOP} total)...")
tl_assets_joint = _build_informed_assets(
    strategy_key="informed_joint",
    cache_subdir="exp_ch_isl_joint",
    force_recompute=False,
)
print(f"informed_joint TL assets: {len(tl_assets_joint)}")

print(f"\nBuilding informed_topo assets "
      f"({len(FRACS)} fracs × {N_TOP} ranks = {len(FRACS)*N_TOP} total)...")
tl_assets_topo = _build_informed_assets(
    strategy_key="informed_topo",
    cache_subdir="exp_ch_isl_topo",
    force_recompute=False,
)
print(f"informed_topo TL assets: {len(tl_assets_topo)}")

# ── Merge all into single dict for downstream fine-tuning ────────────────
tl_assets_exp = {**tl_assets_random, **tl_assets_joint, **tl_assets_topo}
print(f"\nTotal TL assets: {len(tl_assets_exp)}")
print("Sample keys:")
for k in list(tl_assets_exp.keys())[:6]:
    print(f"  {k}")

### LSTM parameters:

In [ ]:
log_path_gs_results = {
    "ISL":
    GLOB_DIR / 'GS_results/lstm_param_search_progress_OOS_ISL_2026-02-11.csv',
    "NOR":
    GLOB_DIR / 'GS_results/lstm_param_search_progress_OOS_NOR_2026-02-09.csv',
    "FR":
    GLOB_DIR / 'GS_results/lstm_param_search_progress_OOS_FR_2026-02-06.csv',
    "CH": GLOB_DIR / 'GS_results/lstm_param_search_progress_CH_2026-02-18.csv',
}

default_params = {
    'Fm': 8,
    'Fs': 3,
    'hidden_size': 128,
    'num_layers': 1,
    'bidirectional': False,
    'dropout': 0.0,
    'static_layers': 1,
    'static_hidden': 128,
    'static_dropout': 0.1,
    'lr': 0.001,
    'weight_decay': 1e-05,
    'loss_name': 'neutral',
    'two_heads': False,
    'head_dropout': 0.1,
    'loss_spec': None
}

params_by_key = build_lstm_params_by_key(
    default_params=default_params,
    log_path_gs_results=log_path_gs_results,
    RGI_REGIONS=RGI_REGIONS,
)


def _params_key_for_source(src_region, params_by_key, default_params):
    """Look up best params for a source region using RGI_REGIONS mapping."""
    for rgi, info in RGI_REGIONS.items():
        if info["code"] == src_region:
            key = f"{rgi}_{src_region}"
            return params_by_key.get(key, default_params)
    # CEU has no single RGI — fall back to default
    return default_params


params_by_key.keys()

### Train or load CH baseline:

In [ ]:
# ── Step D: Train or load CH baseline ────────────────────────────────────
# We only need CH as the source for this experiment.

TRAIN_FLAG = False  # set True to retrain from scratch
MODEL_DATE = "2026-05-11"

ch_baseline_model, ch_baseline_path, ch_baseline_info = train_or_load_CH_baseline(
    cfg=cfg,
    tl_assets=
    tl_assets_exp,  # use exp assets — any key works, just needs the train split
    default_params=_params_key_for_source("CH", params_by_key, default_params),
    device=device,
    models_dir=BASE_DIR / "models" / "CH",
    prefix="lstm_CH",
    key="defaultparams",
    train_flag=TRAIN_FLAG,
    force_retrain=False,
    epochs=150,
    verbose=False,
    date=MODEL_DATE,
)

print(f"CH baseline loaded from: {ch_baseline_path}")

### Fine tune LSTM:

### Adapters:

In [ ]:
# load GS results:
import json

with open(
        GLOB_DIR /
        'GS_results/tuning_adapter/adapter_best_by_region_2026-02-24_15.json',
        "r") as f:
    best_by_region = json.load(f)

# ── Step E: Fine-tune with adapters for all CH→ISL subsets ───────────────

FORCE_RETRAIN = False
FT_DATE = "2026-05-11"

exp_adapter_models = {}
exp_adapter_infos = {}

exp_adapter_models, exp_adapter_infos = finetune_TL_models_all(
    cfg=cfg,
    tl_assets_by_key=tl_assets_exp,
    best_params=_params_key_for_source("CH", params_by_key, default_params),
    device=device,
    pretrained_ckpt_path=ch_baseline_path,
    strategies=["adapter"],
    force_retrain=FORCE_RETRAIN,
    prefix="lstm_CH_to_ISL_exp",
    verbose=False,
    best_by_region=best_by_region,
    models_dir=BASE_DIR / "models" / "CH" / "exp_ch_isl_2",
    date=FT_DATE,
)

print(f"Fine-tuned model keys ({len(exp_adapter_models)}):")
for k in exp_adapter_models:
    print(f"  {k}")

### DAN:

In [ ]:
# ── Step E2: Fine-tune with DAN for all CH→ISL subsets ───────────────────

FORCE_RETRAIN_DAN = False
DAN_DATE = "2026-05-11"

# DAN's regions_only expects the target region labels as they appear in the
# asset keys — extract them the same way the original code does
dan_regions = sorted(
    set(
        k.split("TL_CH_to_")[1].rsplit("_", 1)[0]
        for k in tl_assets_exp.keys()))
print(f"DAN target regions: {dan_regions}")

exp_dan_models, exp_dan_infos = finetune_TL_models_all(
    cfg=cfg,
    tl_assets_by_key=tl_assets_exp,
    best_params=_params_key_for_source("CH", params_by_key, default_params),
    device=device,
    pretrained_ckpt_path=ch_baseline_path,
    strategies=["dan"],
    force_retrain=FORCE_RETRAIN_DAN,
    prefix="lstm_CH_to_ISL_exp",
    verbose=False,
    regions_only=dan_regions,
    best_by_region=best_by_region,
    models_dir=BASE_DIR / "models" / "CH" / "exp_ch_isl_2",
    date=DAN_DATE,
)

print(f"DAN model keys ({len(exp_dan_models)}):")
for k in exp_dan_models:
    print(f"  {k}")

### Evaluate on test:

In [ ]:
# ── Step F: Evaluate all fine-tuned models on fixed ISL test set ──────────

results = []  # list of dicts, one per subset

for frac in FRACS:
    pct = int(frac * 100)

    # --- Informed ---
    asset_key = f"TL_CH_to_ISL_ISL_informed_{pct}pct"  # adjust if printed keys differ
    model_key = [k for k in exp_adapter_models if f"informed_{pct}pct" in k]

    if not model_key:
        print(f"WARNING: no model found for informed_{pct}pct, skipping")
    else:
        model_key = model_key[0]
        ax = plt.subplot(1, 1, 1)
        plt.close()  # we don't need the plots, just the metrics

        metrics, df_preds, _, _ = evaluate_one_model_TL(
            cfg=cfg,
            model=exp_adapter_models[model_key],
            device=device,
            tl_assets_for_key=tl_assets_exp[asset_key],
            ax=ax,
            ax_xlim=None,
            ax_ylim=None,
            title=None,
            legend_fontsize=10,
            batch_size=128,
            domain_vocab=None,
            show_legend=False,
        )
        results.append({
            "strategy":
            "informed",
            "frac":
            frac,
            "pct":
            pct,
            "seed_idx":
            None,
            "n_glaciers":
            ft_subsets["informed"][str(pct)]["n_glaciers"],
            "n_meas":
            ft_subsets["informed"][str(pct)]["n_meas"],
            **metrics,
        })
        print(f"  informed @ {pct}%: {metrics}")

    # --- Random seeds ---
    for seed_idx in range(N_SEEDS):
        asset_key = f"TL_CH_to_ISL_ISL_random_{pct}pct_seed{seed_idx}"
        model_key = [
            k for k in exp_adapter_models
            if f"random_{pct}pct_seed{seed_idx}" in k
        ]

        if not model_key:
            print(
                f"WARNING: no model found for random_{pct}pct_seed{seed_idx}, skipping"
            )
            continue
        model_key = model_key[0]

        ax = plt.subplot(1, 1, 1)
        plt.close()

        metrics, df_preds, _, _ = evaluate_one_model_TL(
            cfg=cfg,
            model=exp_adapter_models[model_key],
            device=device,
            tl_assets_for_key=tl_assets_exp[asset_key],
            ax=ax,
            ax_xlim=None,
            ax_ylim=None,
            title=None,
            legend_fontsize=10,
            batch_size=128,
            domain_vocab=None,
            show_legend=False,
        )
        results.append({
            "strategy":
            "random",
            "frac":
            frac,
            "pct":
            pct,
            "seed_idx":
            seed_idx,
            "n_glaciers":
            ft_subsets["random"][str(pct)][seed_idx]["n_glaciers"],
            "n_meas":
            ft_subsets["random"][str(pct)][seed_idx]["n_meas"],
            **metrics,
        })

    print(f"  random @ {pct}%: done ({N_SEEDS} seeds)")

df_results = pd.DataFrame(results)
df_results.to_csv(BASE_DIR / "exp_ch_isl_results.csv", index=False)
print(f"\nResults saved — {len(df_results)} rows")
print(df_results.groupby(["strategy", "pct"]).mean(numeric_only=True).round(3))

In [ ]:
# ── Evaluate CH baseline (no FT) on ISL test set ─────────────────────────

ax_dummy = plt.subplot(1, 1, 1)
plt.close()

# use any asset key that has the ISL test data
any_isl_key = next(k for k in tl_assets_exp if "informed_5pct" in k)

baseline_metrics, _, _, _ = evaluate_one_model_TL(
    cfg=cfg,
    model=ch_baseline_model,
    device=device,
    tl_assets_for_key=tl_assets_exp[any_isl_key],
    ax=ax_dummy,
    ax_xlim=None,
    ax_ylim=None,
    title=None,
    legend_fontsize=10,
    batch_size=128,
    domain_vocab=None,
    show_legend=False,
)
print("CH baseline (no FT) metrics:", baseline_metrics)

In [ ]:
import matplotlib.ticker as mtick

# ── Step G2: Combined plot — informed overlaid on random curve ────────────
METRICS = ["RMSE_annual", "RMSE_winter", "R2_annual", "R2_winter"]
FRACS_PLOT = sorted(df_results["frac"].unique())
PCTS_PLOT = [int(f * 100) for f in FRACS_PLOT]

fig, axes = plt.subplots(
    nrows=2,
    ncols=2,
    figsize=(12, 8),
    sharex=True,
)
axes = axes.flatten()

for ax, metric in zip(axes, METRICS):
    is_r2 = metric.startswith("R2")

    # ── Random: mean ± std band + individual seed dots ──────────────────
    df_rnd = df_results[df_results["strategy"] == "random"]
    agg = (df_rnd.groupby("pct")[metric].agg(["mean",
                                              "std"]).reindex(PCTS_PLOT))
    ax.plot(PCTS_PLOT,
            agg["mean"],
            marker="o",
            color="steelblue",
            label="Random (mean)",
            zorder=4)
    ax.fill_between(
        PCTS_PLOT,
        agg["mean"] - agg["std"],
        agg["mean"] + agg["std"],
        alpha=0.2,
        color="steelblue",
        label="Random ±1 std",
    )
    for _, row_r in df_rnd.iterrows():
        ax.scatter(row_r["pct"],
                   row_r[metric],
                   color="steelblue",
                   alpha=0.25,
                   s=15,
                   zorder=3)

    # ── Informed: one point per fraction ────────────────────────────────
    df_inf = df_results[df_results["strategy"] == "informed"]
    inf_vals = df_inf.groupby("pct")[metric].mean().reindex(PCTS_PLOT)
    ax.plot(PCTS_PLOT,
            inf_vals,
            marker="D",
            linestyle="--",
            color="darkorange",
            label="Informed (Sinkhorn)",
            zorder=5,
            markersize=8)

    # ── Formatting ───────────────────────────────────────────────────────
    ax.set_title(metric, fontsize=11)
    ax.set_ylabel(metric, fontsize=10)
    ax.set_xticks(PCTS_PLOT)
    ax.xaxis.set_major_formatter(mtick.FormatStrFormatter("%d%%"))
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)
    ax.set_xlabel("FT data (% of total ISL)", fontsize=10)

    # shade the region where informed beats random
    mean_rnd = agg["mean"].values
    inf_arr = inf_vals.values
    x = np.array(PCTS_PLOT)

    if is_r2:
        # informed better = higher R2
        better_mask = inf_arr > mean_rnd
    else:
        # informed better = lower RMSE
        better_mask = inf_arr < mean_rnd

    ax.fill_between(
        x,
        mean_rnd,
        inf_arr,
        where=better_mask,
        alpha=0.15,
        color="green",
        label="Informed better",
    )
    ax.fill_between(
        x,
        mean_rnd,
        inf_arr,
        where=~better_mask,
        alpha=0.15,
        color="red",
        label="Random better",
    )

    # ── CH baseline (no FT) ──────────────────────────────────────────────
    ax.axhline(
        baseline_metrics[metric],
        color="grey",
        linestyle=":",
        linewidth=1.5,
        label="CH baseline (no FT)",
        zorder=2,
    )

    ax.legend(fontsize=8)

fig.suptitle(
    "CH → ISL  |  Adapter fine-tuning: Informed vs. Random sampling\n"
    f"Fixed test set: {len(ISL_TEST_SET)} glaciers | {N_SEEDS} random seeds",
    fontsize=13,
)
plt.tight_layout()
fig.savefig(BASE_DIR / "exp_ch_isl_combined.pdf", bbox_inches="tight")
fig.savefig(BASE_DIR / "exp_ch_isl_combined.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved.")